In [1]:
import numpy as np
import pandas as pd

from ta.trend import EMAIndicator, MACD
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.volatility import AverageTrueRange, BollingerBands

print("All imports successful!")
print(np.__version__)
print(pd.__version__)

All imports successful!
1.26.4
2.2.2


In [5]:
# ============================================================
#  PHASE 2 — DATA CLEANING + 18 TECHNICAL INDICATORS
#  Stock Price Prediction & Analysis System
# ============================================================
# Prerequisites: phase1_data_ingestion.py must be run first
# Input : data/raw/master_raw.csv
# Output: data/processed/master_features.csv
# ============================================================

# ── CELL 1: Install & Imports ────────────────────────────────
# !pip install pandas numpy ta scikit-learn tqdm

import pandas as pd
import numpy as np
from ta.trend     import EMAIndicator, MACD
from ta.momentum  import RSIIndicator, StochasticOscillator
from ta.volatility import AverageTrueRange, BollingerBands
from ta.volume    import OnBalanceVolumeIndicator, VolumeWeightedAveragePrice
from sklearn.preprocessing import RobustScaler
import warnings, os
from tqdm import tqdm
warnings.filterwarnings("ignore")

PROCESSED_DIR = "data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# ── CELL 2: Load Raw Data ────────────────────────────────────
master = pd.read_csv(
    "data/raw/master_raw.csv",
    parse_dates=["index"]
)
master.rename(columns={"index": "Date"}, inplace=True)
master.sort_values(["Ticker","Date"], inplace=True)
master.reset_index(drop=True, inplace=True)

print(f"Loaded  : {len(master):,} rows | {master['Ticker'].nunique()} tickers")
print(f"Columns : {list(master.columns)}")
print(f"Missing : {master.isnull().sum().sum():,} values")

# ── CELL 3: Cleaning Pipeline ────────────────────────────────
def clean_ticker_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Per-ticker cleaning:
    1. Remove duplicate dates
    2. Reindex to full trading calendar (fill gaps)
    3. Forward-fill then back-fill prices (max 5 consecutive)
    4. Drop rows where Close is still null
    5. Remove zero-volume days
    """
    df = df.drop_duplicates(subset="Date")
    df = df.set_index("Date").sort_index()

    # Reindex to business days
    full_idx = pd.bdate_range(df.index.min(), df.index.max())
    df = df.reindex(full_idx)
    df.index.name = "Date"

    # Fill price columns
    price_cols = ["Open","High","Low","Close"]
    df[price_cols] = df[price_cols].fillna(method="ffill", limit=5)
    df[price_cols] = df[price_cols].fillna(method="bfill", limit=5)

    # Fill volume with 0 on padded days, then ticker/cap from forward fill
    df["Volume"]   = df["Volume"].fillna(0)
    df["Ticker"]   = df["Ticker"].fillna(method="ffill")
    df["Cap_Type"] = df["Cap_Type"].fillna(method="ffill")

    # Drop remaining nulls in Close
    df = df.dropna(subset=["Close"])

    # Remove sessions with zero close price
    df = df[df["Close"] > 0]
    df = df[df["Volume"] >= 0]

    df.reset_index(inplace=True)
    return df

print("\nCleaning each ticker...")
cleaned_dfs = []
for ticker, grp in tqdm(master.groupby("Ticker"), desc="Cleaning"):
    cleaned_dfs.append(clean_ticker_df(grp.copy()))

clean_master = pd.concat(cleaned_dfs, ignore_index=True)
clean_master.sort_values(["Ticker","Date"], inplace=True)
print(f"After cleaning : {len(clean_master):,} rows | missing: {clean_master.isnull().sum().sum()}")

# ── CELL 4: 18-Indicator Feature Engine ─────────────────────
#
#  Category     │ Indicators
#  ─────────────┼──────────────────────────────────────────────
#  Trend (5)    │ EMA50, EMA200, MACD, MACD_Signal, EMA_RATIO
#  Momentum (5) │ RSI14, Stoch_K, Stoch_D, ROC10, MOM10
#  Volatility(5)│ ATR14, BB_Width, BB_Upper, BB_Lower, RollingStd20
#  Volume  (3)  │ OBV, VWAP, Vol_Ratio20
#  ─────────────┼──────────────────────────────────────────────
#  Target  (2)  │ Daily_Return, Target (binary 0/1)

def compute_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Compute all 18 technical indicators for one ticker."""
    close  = df["Close"]
    high   = df["High"]
    low    = df["Low"]
    volume = df["Volume"]

    # ── TREND ──────────────────────────────────────────
    ema50  = EMAIndicator(close=close, window=50,  fillna=True)
    ema200 = EMAIndicator(close=close, window=200, fillna=True)
    macd   = MACD(close=close, window_slow=26, window_fast=12, window_sign=9, fillna=True)

    df["EMA50"]       = ema50.ema_indicator()
    df["EMA200"]      = ema200.ema_indicator()
    df["MACD"]        = macd.macd()
    df["MACD_Signal"] = macd.macd_signal()
    df["EMA_RATIO"]   = df["EMA50"] / df["EMA200"]   # >1 = bullish trend

    # ── MOMENTUM ───────────────────────────────────────
    rsi   = RSIIndicator(close=close, window=14, fillna=True)
    stoch = StochasticOscillator(high=high, low=low, close=close,
                                  window=14, smooth_window=3, fillna=True)

    df["RSI14"]   = rsi.rsi()
    df["Stoch_K"] = stoch.stoch()
    df["Stoch_D"] = stoch.stoch_signal()
    df["ROC10"]   = close.pct_change(periods=10) * 100      # Rate of Change
    df["MOM10"]   = close.diff(periods=10)                  # Momentum

    # ── VOLATILITY ─────────────────────────────────────
    atr = AverageTrueRange(high=high, low=low, close=close, window=14, fillna=True)
    bb  = BollingerBands(close=close, window=20, window_dev=2, fillna=True)

    df["ATR14"]        = atr.average_true_range()
    df["BB_Upper"]     = bb.bollinger_hband()
    df["BB_Lower"]     = bb.bollinger_lband()
    df["BB_Width"]     = bb.bollinger_wband()              # (upper-lower)/mid
    df["RollingStd20"] = close.rolling(window=20).std()

    # ── VOLUME ─────────────────────────────────────────
    obv  = OnBalanceVolumeIndicator(close=close, volume=volume, fillna=True)
    df["OBV"]        = obv.on_balance_volume()
    # VWAP (daily rolling proxy)
    df["VWAP"]       = (close * volume).rolling(20).sum() / volume.rolling(20).sum()
    df["Vol_Ratio20"] = volume / volume.rolling(20).mean()  # volume relative to 20d avg

    # ── TARGET VARIABLE ────────────────────────────────
    df["Daily_Return"] = close.pct_change() * 100
    df["Target"]       = (df["Daily_Return"].shift(-1) > 0.35).astype(int)
    # Target = 1 (Bullish): next-day return > 0.35%
    # Target = 0 (Bearish): next-day return ≤ 0.35%

    return df

print("\nComputing 18 indicators for each ticker...")
featured_dfs = []
for ticker, grp in tqdm(clean_master.groupby("Ticker"), desc="Indicators"):
    grp = grp.copy().reset_index(drop=True)
    grp = compute_indicators(grp)
    featured_dfs.append(grp)

featured = pd.concat(featured_dfs, ignore_index=True)
print(f"Feature matrix : {featured.shape}")

# ── CELL 5: Final Cleaning After Indicators ──────────────────
# Drop the last row per ticker (Target is NaN due to shift)
featured = featured[featured["Target"].notna()]

# Drop rows where core indicators are still NaN
# (first ~200 rows per ticker before EMA200 warms up)
FEATURE_COLS = [
    "EMA50","EMA200","MACD","MACD_Signal","EMA_RATIO",
    "RSI14","Stoch_K","Stoch_D","ROC10","MOM10",
    "ATR14","BB_Upper","BB_Lower","BB_Width","RollingStd20",
    "OBV","VWAP","Vol_Ratio20"
]
before = len(featured)
featured = featured.dropna(subset=FEATURE_COLS)
print(f"Dropped {before - len(featured):,} warm-up rows → {len(featured):,} clean rows remain")

# ── CELL 6: Add Cap-Type Encoding ────────────────────────────
cap_map = {"large": 0, "mid": 1, "small": 2}
featured["Cap_Encoded"] = featured["Cap_Type"].map(cap_map)

# ── CELL 7: Data Statistics ──────────────────────────────────
print("\n" + "="*55)
print("  FEATURE DATASET STATISTICS")
print("="*55)
print(f"  Total rows     : {len(featured):,}")
print(f"  Total tickers  : {featured['Ticker'].nunique()}")
print(f"  Features       : {len(FEATURE_COLS)}")
print(f"\n  Target distribution (overall):")
print(featured["Target"].value_counts(normalize=True).round(3).to_string())
print(f"\n  Target by cap type:")
print(featured.groupby("Cap_Type")["Target"].mean().round(3).to_string())
print(f"\n  Missing values  : {featured.isnull().sum().sum()}")
print("="*55)

# ── CELL 8: Save Processed Data ──────────────────────────────
out_path = f"{PROCESSED_DIR}/master_features.csv"
featured.to_csv(out_path, index=False)
print(f"\n✅  Saved → {out_path}")

# Also save per-cap CSVs for easier loading
for cap in ["large","mid","small"]:
    sub = featured[featured["Cap_Type"] == cap]
    sub.to_csv(f"{PROCESSED_DIR}/{cap}_cap_features.csv", index=False)
    print(f"  Saved {cap}_cap_features.csv  ({len(sub):,} rows)")

print("\n➡  Next: Run phase3_xgboost_model.py")

Loaded  : 123,281 rows | 130 tickers
Columns : ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker', 'Cap_Type']
Missing : 0 values

Cleaning each ticker...


Cleaning: 100%|██████████| 130/130 [00:06<00:00, 20.61it/s]


After cleaning : 129,257 rows | missing: 0

Computing 18 indicators for each ticker...


Indicators: 100%|██████████| 130/130 [00:05<00:00, 22.68it/s]


Feature matrix : (129257, 28)
Dropped 2,470 warm-up rows → 126,787 clean rows remain

  FEATURE DATASET STATISTICS
  Total rows     : 126,787
  Total tickers  : 130
  Features       : 18

  Target distribution (overall):
Target
0    0.615
1    0.385

  Target by cap type:
Cap_Type
large    0.383
mid      0.392
small    0.376

  Missing values  : 0

✅  Saved → data/processed/master_features.csv
  Saved large_cap_features.csv  (46,080 rows)
  Saved mid_cap_features.csv  (46,881 rows)
  Saved small_cap_features.csv  (33,826 rows)

➡  Next: Run phase3_xgboost_model.py
